<a href="https://colab.research.google.com/github/xuanbin159/BK_pot_ML/blob/main/2_embedding_autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. 임베딩 생성 및 Autoencoder 차원 축소

본 노트북은 OpenAI API를 활용하여 텍스트 임베딩을 생성하고, PyTorch Autoencoder를 통해 2차원으로 차원을 축소합니다.

In [ ]:
!pip install openai

In [ ]:
import os
import pandas as pd
import openai
import matplotlib.pyplot as plt
import numpy as np
import math
import torch
import torch.nn as nn

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

## 2.1 데이터 로드

In [ ]:
# p_name 설정
p_name = 'TF'

# 전처리된 학습 데이터 로드
save_dir = f'/content/drive/MyDrive/beam_width_results_{p_name}'
df = pd.read_csv(f'{save_dir}/train_df.csv')
use_df = df['results'].astype(str).tolist()
print(f"텍스트 수: {len(use_df)}")

## 2.2 OpenAI 임베딩 생성

In [ ]:
# OpenAI API 키 설정
os.environ["OPENAI_API_KEY"] = ""  # 본인의 API 키 입력

def get_embeddings_openai(text_list, model="text-embedding-ada-002") -> torch.Tensor:
    batches = math.ceil(len(text_list) / 128)
    outputs = []
    for batch in range(batches):
        text_list_batch = text_list[batch * 128 : (batch + 1) * 128]
        response = openai.embeddings.create(
            input=text_list_batch,
            model=model,
            encoding_format="float",
        )
        batch_embeddings = [e.embedding for e in response.data]
        outputs.extend(batch_embeddings)

    embeddings_tensor = torch.tensor(outputs, dtype=torch.float64)
    torch.set_printoptions(precision=10)
    return embeddings_tensor

In [ ]:
# 임베딩 생성 (시간이 오래 걸릴 수 있음)
embeddings = get_embeddings_openai(use_df)
print(f"임베딩 shape: {embeddings.shape}")

In [ ]:
# 임베딩 저장
embeddings_df = pd.DataFrame(embeddings)
embeddings_df.to_csv('/content/drive/MyDrive/embeddings_last.csv', index=False)
print("임베딩 저장 완료")

## 2.3 저장된 임베딩 로드 (선택)

In [ ]:
# 이미 저장된 임베딩을 로드하는 경우
embeddings = pd.read_csv('/content/drive/MyDrive/embeddings_last.csv')
embeddings = embeddings.values
embeddings = torch.tensor(embeddings, dtype=torch.float64)
print(f"임베딩 shape: {embeddings.shape}")

## 2.4 Autoencoder 모델 정의

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_size, encoding_dim):
        super(Autoencoder, self).__init__()

        # 인코더 정의
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.Tanh(),
            nn.Linear(1024, 512),
            nn.Tanh(),
            nn.Linear(512, 256),
            nn.Tanh(),
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 8),
            nn.Tanh(),
            nn.Linear(8, 4),
            nn.Tanh(),
            nn.Linear(4, encoding_dim)
        )

        # 디코더 정의
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 4),
            nn.Tanh(),
            nn.Linear(4, 8),
            nn.Tanh(),
            nn.Linear(8, 16),
            nn.Tanh(),
            nn.Linear(16, 32),
            nn.Tanh(),
            nn.Linear(32, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.Tanh(),
            nn.Linear(128, 256),
            nn.Tanh(),
            nn.Linear(256, 512),
            nn.Tanh(),
            nn.Linear(512, 1024),
            nn.Tanh(),
            nn.Linear(1024, input_size),
            nn.Tanh()
        )

    def forward(self, x):
        encoded = self.encoder(x).to(torch.float64)
        decoded = self.decoder(encoded).to(torch.float64)
        return encoded, decoded

## 2.5 Autoencoder 학습

In [ ]:
# Embeddings의 크기 설정
input_size = embeddings.shape[1]
encoding_dim = 2  # 2차원으로 축소

# 모델, 손실 함수 정의
model = Autoencoder(input_size, encoding_dim)
criterion = nn.MSELoss()

# 모델과 손실 함수를 double precision으로 변환
model = model.double()
criterion = criterion.double()

# 옵티마이저 정의
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.005)

# embeddings를 double precision으로 변환
embeddings = embeddings.to(torch.float64)

# 학습 루프
num_epochs = 100
for epoch in range(num_epochs):
    encoded, decoded = model(embeddings)
    loss = criterion(decoded, embeddings)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f'Epoch {epoch}/{num_epochs}, Loss: {loss.item()}')

In [ ]:
# 모델 저장
torch.save(model, '/content/drive/MyDrive/last.pth')
print("모델 저장 완료")

## 2.6 저장된 모델 로드 (선택)

In [ ]:
# 전체 모델을 로드하는 경우
model = torch.load('/content/drive/MyDrive/last.pth')

## 2.7 인코딩 결과 시각화

In [ ]:
with torch.no_grad():
    encoded_embeddings, decoded_embeddings = model(embeddings)

In [ ]:
x = encoded_embeddings[:, 0].numpy()
y = encoded_embeddings[:, 1].numpy()

plt.figure(figsize=(10, 8))
plt.scatter(x, y, alpha=0.5)
plt.title('Encoded Embeddings Visualization')
plt.xlabel('First Dimension')
plt.ylabel('Second Dimension')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 인코딩된 임베딩 저장 (다음 노트북에서 사용)
encoded_df = pd.DataFrame(encoded_embeddings.numpy(), columns=['dim1', 'dim2'])
encoded_df.to_csv(f'/content/drive/MyDrive/beam_width_results_{p_name}/encoded_embeddings.csv', index=False)
print("인코딩된 임베딩 저장 완료")